In [ ]:
!pip install segmentation-models-pytorch albumentations torchmetrics pydicom nibabel grad_cam
!wget https://github.com/pitthexai/AISummerSchoolinMedicalImagingInformatics/raw/refs/heads/main/2026_Materials/Day5_Materials/PittAISummerSchool_2026_JSW-Dataset.zip

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import copy

from zipfile import ZipFile

from io import BytesIO
from gzip import GzipFile

import os

import cv2
from PIL import Image

import pydicom
import nibabel
from nibabel import FileHolder, Nifti1Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from segmentation_models_pytorch import utils as smp_utils

from torchmetrics.segmentation import DiceScore, MeanIoU
import torchvision.transforms.functional as TF

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Dataset Setup

In [ ]:
directory = "PittAISummerSchool_2026_JSW-Dataset"
zipfile = "PittAISummerSchool_2026_JSW-Dataset.zip"

zipfile_loc = f"/content/{zipfile}"
data_save_location = f"/content/{directory}/"

In [ ]:
if not os.path.exists(data_save_location):
    with ZipFile(zipfile_loc, 'r') as zipf:
        zipf.extractall("/content/")

In [ ]:
def build_dataset_dataframe(train_x_path, train_y_path):
    pairs = []
    for file in os.listdir(train_x_path):
        train_x_path_file = os.path.join(train_x_path, file)
        train_y_path_file = os.path.join(train_y_path, file)
        pairs.append((train_x_path_file, train_y_path_file))
    df = pd.DataFrame(pairs, columns=['image', 'mask'])
    return df

In [ ]:
df = build_dataset_dataframe(f"{data_save_location}/Train_X_500", f"{data_save_location}/Train_Y_500")